# EDA cho Vietnamese ABSA dataset

Notebook nay doc `data/train.jsonl`, `data/dev.jsonl`, `data/test.jsonl`, thong ke dataset, hien thi cac hinh EDA trong notebook va luu hinh vao `figures/dataset/`.


In [ ]:
from __future__ import annotations

import json
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        return None


def find_project_root() -> Path:
    """Find the folder that contains data/*.jsonl from common notebook locations."""
    candidates = [Path.cwd(), Path.cwd().parent]
    if len(Path.cwd().parents) > 1:
        candidates.append(Path.cwd().parents[1])
    for candidate in candidates:
        if all((candidate / "data" / f"{split}.jsonl").exists() for split in ["train", "dev", "test"]):
            return candidate.resolve()
    raise FileNotFoundError("Cannot find data/train.jsonl, data/dev.jsonl, and data/test.jsonl.")


ROOT = find_project_root()
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "figures" / "dataset"

ASPECT_ORDER = [
    "GENERAL",
    "PERFORMANCE",
    "BATTERY",
    "FEATURES",
    "CAMERA",
    "SER&ACC",
    "DESIGN",
    "PRICE",
    "SCREEN",
    "STORAGE",
]
SENTIMENT_ORDER = ["POSITIVE", "NEGATIVE", "NEUTRAL"]

COLORS = {
    "orange": "#FF6B35",
    "blue": "#1F77B4",
    "green": "#2E7D32",
    "red": "#B3261E",
    "yellow": "#F2B705",
    "gray": "#5C5C5C",
}
SENTIMENT_COLORS = {
    "POSITIVE": COLORS["green"],
    "NEGATIVE": COLORS["red"],
    "NEUTRAL": COLORS["blue"],
}


def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def fmt(n: float | int) -> str:
    return f"{int(round(n)):,}".replace(",", ".")


def pct(n: int, total: int, digits: int = 1) -> str:
    return f"{100 * n / total:.{digits}f}%"


def prepare_output_dir() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    for pattern in ("*.png", "*.svg"):
        for path in OUT_DIR.glob(pattern):
            path.unlink()


def save_fig(fig: plt.Figure, name: str) -> None:
    fig.savefig(OUT_DIR / f"{name}.png", dpi=220, bbox_inches="tight", facecolor="white")
    display(fig)
    plt.close(fig)


def set_center_title(ax: plt.Axes, title: str) -> None:
    ax.set_title(title, fontsize=20, fontweight="bold", pad=18, loc="center")


def style_axes(ax: plt.Axes, *, grid_axis: str = "x") -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#C8CCD2")
    ax.spines["bottom"].set_color("#C8CCD2")
    ax.tick_params(colors="#333333", labelsize=11)
    ax.grid(axis=grid_axis, color="#E5E7EB", linewidth=0.8)
    ax.set_axisbelow(True)


def load_records() -> tuple[pd.DataFrame, pd.DataFrame]:
    review_rows = []
    label_rows = []

    for split in ["train", "dev", "test"]:
        for idx, row in enumerate(read_jsonl(DATA_DIR / f"{split}.jsonl")):
            labels = row.get("labels", [])
            text = row["text"]
            aspects_in_review = {label.split("#")[0] for _, _, label in labels}
            review_rows.append(
                {
                    "split": split,
                    "review_id": f"{split}-{idx}",
                    "text": text,
                    "text_len": len(text),
                    "num_spans": len(labels),
                    "num_aspects": len(aspects_in_review),
                    "nfc_changed": unicodedata.normalize("NFC", text) != text,
                }
            )
            for start, end, label in labels:
                aspect, sentiment = label.split("#")
                span = text[start:end]
                label_rows.append(
                    {
                        "split": split,
                        "review_id": f"{split}-{idx}",
                        "start": start,
                        "end": end,
                        "label": label,
                        "aspect": aspect,
                        "sentiment": sentiment,
                        "span_text": span,
                        "span_len": end - start,
                        "has_outer_whitespace": span != span.strip(),
                    }
                )

    return pd.DataFrame(review_rows), pd.DataFrame(label_rows)


def figure_aspects_per_review(reviews: pd.DataFrame) -> None:
    counts = reviews["num_aspects"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(9.8, 5.6))
    bars = ax.bar(counts.index.astype(str), counts.values, color=COLORS["orange"])
    set_center_title(ax, "Number of Aspects per Review")
    ax.set_xlabel("Number of distinct aspects")
    ax.set_ylabel("Number of reviews")
    style_axes(ax, grid_axis="y")
    ax.grid(axis="x", visible=False)
    ax.bar_label(bars, labels=[fmt(v) for v in counts.values], padding=3, fontsize=10.5)
    fig.tight_layout()
    save_fig(fig, "01_aspects_per_review")


def figure_aspect_distribution(labels: pd.DataFrame) -> None:
    counts = labels["aspect"].value_counts().reindex(ASPECT_ORDER)
    fig, ax = plt.subplots(figsize=(10.4, 6.2))
    bars = ax.barh(counts.index[::-1], counts.values[::-1], color=COLORS["orange"])
    set_center_title(ax, "Aspect Distribution")
    ax.set_xlabel("Number of spans")
    ax.set_ylabel("Aspect")
    style_axes(ax)
    for bar, value in zip(bars, counts.values[::-1]):
        ax.text(value + counts.max() * 0.012, bar.get_y() + bar.get_height() / 2, fmt(value), va="center", fontsize=10.5)
    ax.set_xlim(0, counts.max() * 1.16)
    fig.tight_layout()
    save_fig(fig, "02_aspect_distribution")


def figure_aspect_sentiment_heatmap(labels: pd.DataFrame) -> None:
    pivot = (
        pd.crosstab(labels["aspect"], labels["sentiment"])
        .reindex(index=ASPECT_ORDER, columns=SENTIMENT_ORDER)
        .fillna(0)
        .astype(int)
    )
    fig, ax = plt.subplots(figsize=(9.6, 6.9))
    sns.heatmap(
        pivot,
        annot=True,
        fmt="d",
        cmap=sns.light_palette(COLORS["orange"], as_cmap=True),
        linewidths=1,
        linecolor="white",
        cbar_kws={"label": "Number of spans"},
        ax=ax,
    )
    set_center_title(ax, "Aspect-Sentiment Heatmap")
    ax.set_xlabel("Sentiment")
    ax.set_ylabel("Aspect")
    ax.tick_params(axis="both", labelsize=10.5)
    fig.tight_layout()
    save_fig(fig, "03_aspect_sentiment_heatmap")


def figure_aspect_sentiment_stack(labels: pd.DataFrame) -> None:
    table = (
        pd.crosstab(labels["aspect"], labels["sentiment"])
        .reindex(index=ASPECT_ORDER, columns=SENTIMENT_ORDER)
        .fillna(0)
        .astype(int)
    )
    fig, ax = plt.subplots(figsize=(10.5, 6.2))
    left = pd.Series(0, index=table.index)
    for sentiment in SENTIMENT_ORDER:
        values = table[sentiment]
        ax.barh(table.index[::-1], values[::-1], left=left[::-1], color=SENTIMENT_COLORS[sentiment], label=sentiment)
        left += values
    set_center_title(ax, "Aspect-Sentiment Stacked Bar Chart")
    ax.set_xlabel("Number of spans")
    ax.set_ylabel("Aspect")
    style_axes(ax)
    ax.legend(title="Sentiment", frameon=False, loc="lower right")
    totals = table.sum(axis=1)
    for y, aspect in enumerate(table.index[::-1]):
        value = int(totals[aspect])
        ax.text(value + totals.max() * 0.01, y, fmt(value), va="center", fontsize=10)
    ax.set_xlim(0, totals.max() * 1.14)
    fig.tight_layout()
    save_fig(fig, "04_aspect_sentiment_stacked_bar")


def figure_sentiment_distribution(labels: pd.DataFrame) -> None:
    counts = labels["sentiment"].value_counts().reindex(SENTIMENT_ORDER)
    fig, ax = plt.subplots(figsize=(8.6, 5.4))
    bars = ax.bar(counts.index, counts.values, color=[SENTIMENT_COLORS[s] for s in counts.index])
    set_center_title(ax, "Sentiment Distribution")
    ax.set_xlabel("Sentiment")
    ax.set_ylabel("Number of spans")
    style_axes(ax, grid_axis="y")
    ax.grid(axis="x", visible=False)
    labels_text = [f"{fmt(v)} ({pct(int(v), int(counts.sum()))})" for v in counts.values]
    ax.bar_label(bars, labels=labels_text, padding=3, fontsize=10.5)
    fig.tight_layout()
    save_fig(fig, "05_sentiment_distribution")


def make_binned_counts(values: pd.Series, bins: list[int], labels: list[str]) -> pd.Series:
    return pd.cut(values, bins=bins, right=False, labels=labels).value_counts().reindex(labels)


def figure_review_length_distribution(reviews: pd.DataFrame) -> None:
    max_len = int(reviews["text_len"].max())
    bins = [0, 100, 200, 300, 400, 500, max_len + 1]
    labels = ["0-99", "100-199", "200-299", "300-399", "400-499", "500+"]
    counts = make_binned_counts(reviews["text_len"], bins, labels)

    fig, ax = plt.subplots(figsize=(9.8, 5.6))
    bars = ax.bar(counts.index.astype(str), counts.values, color=COLORS["blue"])
    set_center_title(ax, "Review Length Distribution")
    ax.set_xlabel("Number of characters")
    ax.set_ylabel("Number of reviews")
    style_axes(ax, grid_axis="y")
    ax.grid(axis="x", visible=False)
    ax.bar_label(bars, labels=[fmt(v) for v in counts.values], padding=3, fontsize=10.5)
    fig.tight_layout()
    save_fig(fig, "06_review_length_distribution")


def figure_spans_per_review_distribution(reviews: pd.DataFrame) -> None:
    counts = reviews["num_spans"].value_counts().sort_index()
    display = counts[counts.index <= 9].copy()
    display.loc[10] = counts[counts.index >= 10].sum()
    labels = [str(i) for i in range(10)] + ["10+"]
    values = [int(display.get(i, 0)) for i in range(10)] + [int(display.get(10, 0))]

    fig, ax = plt.subplots(figsize=(9.8, 5.6))
    bars = ax.bar(labels, values, color=COLORS["orange"])
    set_center_title(ax, "Spans per Review Distribution")
    ax.set_xlabel("Number of spans")
    ax.set_ylabel("Number of reviews")
    style_axes(ax, grid_axis="y")
    ax.grid(axis="x", visible=False)
    ax.bar_label(bars, labels=[fmt(v) for v in values], padding=3, fontsize=10.5)
    fig.tight_layout()
    save_fig(fig, "07_spans_per_review_distribution")


def figure_span_length_distribution(labels: pd.DataFrame) -> None:
    max_len = int(labels["span_len"].max())
    bins = [0, 20, 40, 60, 80, 100, 120, 140, 160, max_len + 1]
    bin_labels = ["0-19", "20-39", "40-59", "60-79", "80-99", "100-119", "120-139", "140-159", "160+"]
    counts = make_binned_counts(labels["span_len"], bins, bin_labels)

    fig, ax = plt.subplots(figsize=(10.2, 5.6))
    bars = ax.bar(counts.index.astype(str), counts.values, color=COLORS["green"])
    set_center_title(ax, "Span Length Distribution")
    ax.set_xlabel("Number of characters")
    ax.set_ylabel("Number of spans")
    style_axes(ax, grid_axis="y")
    ax.grid(axis="x", visible=False)
    ax.bar_label(bars, labels=[fmt(v) for v in counts.values], padding=3, fontsize=10.5)
    fig.tight_layout()
    save_fig(fig, "08_span_length_distribution")


def figure_annotation_noise_summary(reviews: pd.DataFrame, labels: pd.DataFrame) -> None:
    values = pd.Series(
        {
            "Span with outer whitespace": int(labels["has_outer_whitespace"].sum()),
            "Review changed by NFC": int(reviews["nfc_changed"].sum()),
            "Review without span": int((reviews["num_spans"] == 0).sum()),
        }
    )

    fig, ax = plt.subplots(figsize=(9.8, 5.4))
    bars = ax.barh(values.index[::-1], values.values[::-1], color=[COLORS["yellow"], COLORS["blue"], COLORS["orange"]])
    set_center_title(ax, "Annotation and Data Noise Summary")
    ax.set_xlabel("Count")
    ax.set_ylabel("")
    style_axes(ax)
    max_v = values.max()
    denominators = {
        "Span with outer whitespace": len(labels),
        "Review changed by NFC": len(reviews),
        "Review without span": len(reviews),
    }
    for bar, label, value in zip(bars, values.index[::-1], values.values[::-1]):
        ax.text(value + max_v * 0.03, bar.get_y() + bar.get_height() / 2, f"{fmt(value)} ({pct(int(value), denominators[label])})", va="center", fontsize=10.5)
    ax.set_xlim(0, max_v * 1.35)
    fig.tight_layout()
    save_fig(fig, "09_annotation_noise_summary")


def figure_preprocessing_impact(reviews: pd.DataFrame, labels: pd.DataFrame) -> None:
    values = pd.Series(
        {
            "Reviews changed by NFC": int(reviews["nfc_changed"].sum()),
            "Spans trimmed": int(labels["has_outer_whitespace"].sum()),
            "Labels dropped": 0,
        }
    )
    colors = [COLORS["blue"], COLORS["orange"], COLORS["gray"]]

    fig, ax = plt.subplots(figsize=(9.8, 5.4))
    bars = ax.bar(values.index, values.values, color=colors)
    set_center_title(ax, "Preprocessing Impact")
    ax.set_xlabel("Preprocessing effect")
    ax.set_ylabel("Count")
    style_axes(ax, grid_axis="y")
    ax.grid(axis="x", visible=False)

    label_text = [
        f"{fmt(values['Reviews changed by NFC'])} ({pct(int(values['Reviews changed by NFC']), len(reviews))})",
        f"{fmt(values['Spans trimmed'])} ({pct(int(values['Spans trimmed']), len(labels))})",
        "0",
    ]
    ax.bar_label(bars, labels=label_text, padding=3, fontsize=10.5)
    ax.set_ylim(0, max(values.max() * 1.22, 10))
    fig.tight_layout()
    save_fig(fig, "10_preprocessing_impact")


def write_support_files(reviews: pd.DataFrame, labels: pd.DataFrame) -> None:
    sample = reviews[reviews["split"] == "train"].sample(n=50, random_state=42).sort_values("review_id")
    sample[["review_id", "text", "text_len", "num_spans", "num_aspects"]].to_csv(
        OUT_DIR / "manual_50_sample_reviews.csv",
        index=False,
        encoding="utf-8-sig",
    )

    summary = {
        "total_reviews": int(len(reviews)),
        "total_spans": int(len(labels)),
        "split_reviews": {k: int(v) for k, v in reviews["split"].value_counts().reindex(["train", "dev", "test"]).items()},
        "split_spans": {k: int(v) for k, v in labels["split"].value_counts().reindex(["train", "dev", "test"]).items()},
        "aspect": {k: int(v) for k, v in labels["aspect"].value_counts().reindex(ASPECT_ORDER).items()},
        "sentiment": {k: int(v) for k, v in labels["sentiment"].value_counts().reindex(SENTIMENT_ORDER).items()},
        "aspects_per_review": {str(k): int(v) for k, v in reviews["num_aspects"].value_counts().sort_index().items()},
        "multi_aspect_reviews": int((reviews["num_aspects"] > 1).sum()),
        "empty_reviews": int((reviews["num_spans"] == 0).sum()),
        "whitespace_spans": int(labels["has_outer_whitespace"].sum()),
        "nfc_changed_reviews": int(reviews["nfc_changed"].sum()),
        "preprocessing_impact": {
            "reviews_changed_by_nfc": int(reviews["nfc_changed"].sum()),
            "spans_trimmed": int(labels["has_outer_whitespace"].sum()),
            "labels_dropped": 0,
            "label_distribution_changed": False,
        },
    }
    (OUT_DIR / "dataset_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")


def main() -> None:
    prepare_output_dir()
    sns.set_theme(style="whitegrid", font="DejaVu Sans")
    plt.rcParams.update({
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
    })

    reviews, labels = load_records()
    figure_aspects_per_review(reviews)
    figure_aspect_distribution(labels)
    figure_aspect_sentiment_heatmap(labels)
    figure_aspect_sentiment_stack(labels)
    figure_sentiment_distribution(labels)
    figure_review_length_distribution(reviews)
    figure_spans_per_review_distribution(reviews)
    figure_span_length_distribution(labels)
    figure_annotation_noise_summary(reviews, labels)
    figure_preprocessing_impact(reviews, labels)
    write_support_files(reviews, labels)
    print(f"Wrote PNG figures to {OUT_DIR}")



## Thong ke nhanh


In [ ]:
reviews, labels = load_records()
print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("FIGURE_DIR:", OUT_DIR)
print("Reviews:", len(reviews))
print("Spans:", len(labels))

display(reviews.head())
display(labels.head())

for name, table in {
    "reviews_by_split": reviews["split"].value_counts().reindex(["train", "dev", "test"]).to_frame("reviews"),
    "spans_by_split": labels["split"].value_counts().reindex(["train", "dev", "test"]).to_frame("spans"),
    "aspects": labels["aspect"].value_counts().reindex(ASPECT_ORDER).to_frame("spans"),
    "sentiments": labels["sentiment"].value_counts().reindex(SENTIMENT_ORDER).to_frame("spans"),
}.items():
    print("\n" + name)
    display(table)

print("\nMulti-aspect reviews:", int((reviews["num_aspects"] > 1).sum()))
print("Reviews without span:", int((reviews["num_spans"] == 0).sum()))
print("Reviews changed by NFC:", int(reviews["nfc_changed"].sum()))
print("Spans with outer whitespace:", int(labels["has_outer_whitespace"].sum()))


## Tao va luu hinh EDA


In [ ]:
main()

print("Saved files:")
for path in sorted(OUT_DIR.iterdir()):
    print("-", path.name)
